# Predicting the payment method for a NYC yellow taxi trip

| Section | What it does | Owner |
|---|---|---|
| 0 · Setup | Libraries, settings, which months to use, get and check the data files | Bri |
| 1 · Data cleaning | Label each trip card or cash, clean out bad trips, prepare each month in parallel | Bri |
| 2 · Features | Turn trips into numbers the model can use | |
| 3 · Models | Baseline, model search, final model | |
| 4 · Evaluation | Score the final model on the test months, once | |

**How to run it.** Run each section in order.

**Laptop vs Rockhill.** On a laptop, Section 0 picks a small sample of January to March 2026 for testing purposes (to be removed). On Rockhill it picks the full dataset on the server.

## Section 0 · Setup

This section carries out the following:
* Loads all necessary libraries
* Chooses the months from the dataset
* Ensures files are present
* Checks that months have the correct columns
* Describes the raw data

### 0.1 Libraries

In [20]:
# If a library is missing, remove the # on the next line, run this cell once, then restart the kernel
# %pip install pandas pyarrow scikit-learn matplotlib

import os
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(var, "1")

import json
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from joblib import Parallel, delayed

pd.set_option("display.width", 140)

### 0.2 Shared settings
Settings every section uses. `ON_ROCKHILL` is `True` when the notebook can see Rockhill's shared data folder, which is how it knows where it is running.

In [21]:
#Rockhills location of the dataset
SHARED_DATA = Path("/home/cs4207/data")
ON_ROCKHILL = SHARED_DATA.exists()

DATA_DIR = Path("data")
PREPARED_DIR = Path("prepared")
RESULTS_DIR = Path("results")
TLC_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet"

#Fixed seed so every run gives the same sampple
SEED = 4207
MAX_WORKERS = 3

#Labels and codes for prediction
CARD, CASH = 1, 2
LABEL = "is_cash"

#Remove leakage
LEAKY_COLUMNS = ["tip_amount", "total_amount", "payment_type"]

for folder in (DATA_DIR, PREPARED_DIR, RESULTS_DIR):
    folder.mkdir(exist_ok=True)
print("Running on", "Rockhill" if ON_ROCKHILL else "a laptop")

Running on a laptop


### 0.3 Input: which data to use
**Months are written `YYYY-MM`.**

Splits the data into train, val and test. 

In [22]:
if ON_ROCKHILL:
    TRAIN = ["2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06"]
    VAL   = ["2026-07"]
    TEST  = ["2026-08", "2026-09"]
    SAMPLE_FRACTION = 0.05
#On laptop, we download 3 months for testing purposes
else:
    TRAIN = ["2026-01"]
    VAL   = ["2026-02"]
    TEST  = ["2026-03"]
    SAMPLE_FRACTION = 0.02
DOWNLOAD_MISSING = True

#Prints information of split
def parse_month(text):
    """'2026-01' -> (2026, 1)"""
    year, month = text.split("-")
    return int(year), int(month)

SPLITS = {"train": [parse_month(m) for m in TRAIN],
          "val":   [parse_month(m) for m in VAL],
          "test":  [parse_month(m) for m in TEST]}
ALL_MONTHS = [ym for months in SPLITS.values() for ym in months]

assert all(SPLITS.values()), "every split needs at least one month"
assert len(ALL_MONTHS) == len(set(ALL_MONTHS)), \
    "a month is in more than one split: that leaks test data into training"
assert 0 < SAMPLE_FRACTION <= 1, "SAMPLE_FRACTION must be between 0 and 1"

for split, months in SPLITS.items():
    print(f"{split:5s} {', '.join(f'{y}-{m:02d}' for y, m in months)}")
print(f"keeping {SAMPLE_FRACTION:.0%} of each month's trips")

train 2026-01
val   2026-02
test  2026-03
keeping 2% of each month's trips


### 0.4 Get the files
For each month, it finds where its data is stored. If there are any months missing, it downloads them from the TLC website. 

In [23]:
#Finds where a months file is
def month_file(year, month):
    name = f"yellow_tripdata_{year}-{month:02d}.parquet"
    for folder in (SHARED_DATA, DATA_DIR):
        if (folder / name).exists():
            return folder / name
    return DATA_DIR / name

#This downloads a months dataset from the TLC website
#Only using this for testing purposes on laptop, to be removed
def download(year, month, dest):
    url = TLC_URL.format(year=year, month=month)
    tmp = dest.with_suffix(".part")
    try:
        with urllib.request.urlopen(url, timeout=60) as response, open(tmp, "wb") as out:
            shutil.copyfileobj(response, out)
        tmp.rename(dest)
    except Exception as err:
        tmp.unlink(missing_ok=True)
        raise RuntimeError(f"Could not download {url} ({err}). Download it in a browser "
                           f"from the TLC page and save it as {dest}") from err

#Loops through every file and records info such as data source, its size and the amount of trips inside it
files = []
for split, months in SPLITS.items():
    for year, month in months:
        path = month_file(year, month)
        source = "Rockhill shared folder" if path.parent == SHARED_DATA else "data/"
        if not path.exists():
            if not DOWNLOAD_MISSING:
                raise FileNotFoundError(f"{path.name} is missing: set DOWNLOAD_MISSING = True")
            print(f"Downloading {path.name} ...", flush=True)
            download(year, month, path)
            source = "downloaded now"
        files.append({"split": split, "month": f"{year}-{month:02d}", "source": source,
                      "MB": round(path.stat().st_size / 1e6, 1),
                      "trips": pq.ParquetFile(path).metadata.num_rows, "path": path})
files = pd.DataFrame(files)

files.drop(columns="path").to_csv(RESULTS_DIR / "00_input_files.csv", index=False)
print(files.drop(columns="path").to_string(index=False))
print(f"\nTotal: {files['trips'].sum():,} trips in {len(files)} files, {files['MB'].sum():,.0f} MB")

split   month source   MB   trips
train 2026-01  data/ 64.2 3724889
  val 2026-02  data/ 58.7 3399866
 test 2026-03  data/ 67.9 3952451

Total: 11,077,206 trips in 3 files, 191 MB


### 0.5 Do the months have the same columns?

In [24]:
#The columns that each month should have
REQUIRED = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "payment_type", "fare_amount",
            "trip_distance", "PULocationID", "DOLocationID"]

#Creats a dictionary of each months columns and column types
columns = pd.DataFrame({f["month"]: {c.name.lower(): str(c.type) for c in pq.read_schema(f["path"])}
                        for _, f in files.iterrows()})

#Ensures no month has a missing column
missing = [c for c in REQUIRED if c.lower() not in columns.index or columns.loc[c.lower()].isna().any()]
assert not missing, f"some months are missing required columns: {missing}"

#Ensures no columns differ from other months
differs = columns[columns.nunique(axis=1, dropna=False) > 1]
print(f"{len(columns)} columns; every required one is in every month")
if differs.empty:
    print("Every month has the same columns and types")
else:
    print("Columns that differ between months (blank = that month doesn't have it):")
    print(differs.fillna("").to_string())

20 columns; every required one is in every month
Every month has the same columns and types


### 0.6 Raw data visualisation

In [25]:
#Maps each code to its payment type
PAYMENT_NAMES = {0: "Flex Fare", 1: "Credit card", 2: "Cash", 3: "No charge",
                 4: "Dispute", 5: "Unknown", 6: "Voided"}

#Reads through the first 200,000 trips
first_file = pq.ParquetFile(files["path"].iloc[0])
raw = next(first_file.iter_batches(batch_size=200_000)).to_pandas()
print(f"First {len(raw):,} trips of {files['month'].iloc[0]}, all {raw.shape[1]} columns\n")

print("How trips were paid:")
print(raw["payment_type"].map(PAYMENT_NAMES).value_counts(normalize=True).round(3).to_string())
share_missing = raw.isna().mean()
print("\nShare of missing values:")
print(share_missing[share_missing > 0].round(3).to_string())
raw.head()

First 200,000 trips of 2026-01, all 20 columns

How trips were paid:
payment_type
Credit card    0.800
Cash           0.164
Dispute        0.028
No charge      0.008

Share of missing values:
Series([], )


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1,0.97,1,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0,0.90,1,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0,1.40,1,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4,5.58,1,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0,2.16,1,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [26]:
raw.describe().T.round(2)

,count,mean,min,25%,50%,75%,max,std
VendorID,200000.0,1.886255,1.0,2.0,2.0,2.0,7.0,0.768479
tpep_pickup_datetime,200000,2026-01-02 12:20:08.537485,2025-12-31 23:57:29,2026-01-01 17:52:43,2026-01-02 14:53:45,2026-01-03 06:48:01,2026-01-03 19:11:44,NaN
tpep_dropoff_datetime,200000,2026-01-02 12:35:38.545325,2025-12-31 23:57:32,2026-01-01 18:08:00,2026-01-02 15:11:03,2026-01-03 07:05:28,2026-01-05 20:44:12,NaN
passenger_count,200000.0,1.46401,0.0,1.0,1.0,2.0,6.0,0.872008
trip_distance,200000.0,3.935556,0.0,0.99,1.74,3.77,36610.27,82.008964
RatecodeID,200000.0,3.923745,1.0,1.0,1.0,1.0,99.0,16.249721
PULocationID,200000.0,161.655985,1.0,132.0,161.0,230.0,265.0,63.743393
DOLocationID,200000.0,161.27757,1.0,107.0,161.0,233.0,265.0,70.59746
payment_type,200000.0,1.264585,1.0,1.0,1.0,1.0,4.0,0.616061
fare_amount,200000.0,19.944072,-629.5,8.6,12.8,23.3,850.0,22.100838


## Section 1 · Data cleaning

Keeps card and cash trips, adds the label for each, removes trips that can't be real, and saves one cleaned file per month in `prepared/`.

The months are prepared in parallel, one month per worker process. This is the **data parallelism** for this section.

Saves: `prepared/YYYY-MM.parquet`, `results/01_cleaning_rules.csv`, `results/01_prepare_stats.csv`, `results/01_timings.json`, `results/01_cash_share.png`

### 1.1 Leakage
`payment_type` has seven codes. We keep 1 (card) and 2 (cash), and our model predicts cash.

The TLC records tips automatically for card payments, and cash tips aren't recorded. So a tip above zero almost always means card, and `total_amount` includes the tip. The below table shows this, which is why we won't use these as a feature moving forward. 

In [27]:
#Groups trips by their payment type and records how many of these payments have tips
summary = (raw.groupby("payment_type")
              .agg(trips=("fare_amount", "size"),
                   share_with_tip=("tip_amount", lambda tips: (tips > 0).mean()))
              .rename(index=PAYMENT_NAMES))
summary["share_of_trips"] = summary["trips"] / summary["trips"].sum()
print(summary.round(3).to_string())

               trips  share_with_tip  share_of_trips
payment_type                                        
Credit card   159961           0.902           0.800
Cash           32778           0.000           0.164
No charge       1644           0.000           0.008
Dispute         5617           0.000           0.028


### 1.2 Cleaning rules
A trip is kept only if it passes every rule:

| Rule | Why |
|---|---|
| paid by card or cash | the two classes we predict |
| picked up in the file's own month | some files contain a few trips with wrong dates|
| fare between \$3 and \$500 | negative fares are refunds; the meter minimum is about \$3 |
| distance between 0.1 and 100 miles | 0 miles is a meter error |
| duration between 1 and 180 minutes | shorter is a cancelled trip, longer a meter left running |

In [28]:
PICKUP, DROPOFF = "tpep_pickup_datetime", "tpep_dropoff_datetime"

# The columns we use. tip_amount and total_amount are left out.
KEEP_COLUMNS = ["VendorID", PICKUP, DROPOFF, "passenger_count", "trip_distance",
                "RatecodeID", "store_and_fwd_flag", "PULocationID", "DOLocationID",
                "payment_type", "fare_amount", "extra", "mta_tax", "tolls_amount",
                "improvement_surcharge", "congestion_surcharge", "airport_fee",
                "cbd_congestion_fee"]


def read_month(path, columns=KEEP_COLUMNS):
    actual = {name.lower(): name for name in pq.read_schema(path).names}
    present = [c for c in columns if c.lower() in actual]
    df = pd.read_parquet(path, columns=[actual[c.lower()] for c in present])
    df.columns = present
    for c in columns:
        if c not in df:
            df[c] = 0.0
    return df[columns]


def cleaning_rules(df, year, month):
    minutes = (df[DROPOFF] - df[PICKUP]).dt.total_seconds() / 60
    return pd.DataFrame({
        "card_or_cash": df["payment_type"].isin([CARD, CASH]),
        "in_month": (df[PICKUP].dt.year == year) & (df[PICKUP].dt.month == month),
        "fare_3_to_500": df["fare_amount"].between(3, 500),
        "distance_0.1_to_100": df["trip_distance"].between(0.1, 100),
        "duration_1_to_180": minutes.between(1, 180),
    })

### 1.3 Prepare every month: Uses either one process or several
`prepare_month` prepares one month. It is run once per month, first one month after another, then with up to `MAX_WORKERS` months at the same time in separate worker processes, and the two times are compared.

In [29]:
#This prepares each month by reading it, creating a sample, cleaning, labelling and saving
def prepare_month(year, month, path, sample_fraction, seed, out_dir):
    start = time.perf_counter()
    df = read_month(path)
    raw_trips = len(df)
    # fixed random sample: the same rows every run (the seed differs per month)
    df = df.sample(frac=sample_fraction, random_state=seed + year * 100 + month)
    rules = cleaning_rules(df, year, month)
    df = df[rules.all(axis=1)].copy()
    df[LABEL] = (df["payment_type"] == CASH).astype("int8")
    df = df.drop(columns="payment_type").sort_values(PICKUP, kind="stable").reset_index(drop=True)
    df.to_parquet(Path(out_dir) / f"{year}-{month:02d}.parquet", index=False)
    return {"month": f"{year}-{month:02d}", "raw_trips": raw_trips, "sampled": len(rules),
            "kept": len(df), "cash_share": round(float(df[LABEL].mean()), 4),
            "seconds": round(time.perf_counter() - start, 2)}

#Prepares every month using n workers
def prepare_all(n_workers):
    jobs = [delayed(prepare_month)(y, m, month_file(y, m), SAMPLE_FRACTION, SEED, PREPARED_DIR)
            for y, m in ALL_MONTHS]
    return pd.DataFrame(Parallel(n_jobs=n_workers)(jobs))


RUN_SEQUENTIAL = True
timings = {"months": len(ALL_MONTHS), "workers": min(MAX_WORKERS, len(ALL_MONTHS)),
           "sample_fraction": SAMPLE_FRACTION}
if RUN_SEQUENTIAL:
    start = time.perf_counter()
    prepare_all(n_workers=1)
    timings["sequential_s"] = round(time.perf_counter() - start, 2)

start = time.perf_counter()
stats = prepare_all(n_workers=timings["workers"])
timings["parallel_s"] = round(time.perf_counter() - start, 2)
if RUN_SEQUENTIAL:
    timings["speedup"] = round(timings["sequential_s"] / timings["parallel_s"], 2)

stats.to_csv(RESULTS_DIR / "01_prepare_stats.csv", index=False)
(RESULTS_DIR / "01_timings.json").write_text(json.dumps(timings, indent=2))
print(timings)
stats

{'months': 3, 'workers': 3, 'sample_fraction': 0.02, 'sequential_s': 3.24, 'parallel_s': 2.43, 'speedup': 1.33}


,month,raw_trips,sampled,kept,cash_share,seconds
0,2026-01,3724889,74498,49406,0.1158,1.30
1,2026-02,3399866,67997,44888,0.1141,1.20
2,2026-03,3952451,79049,57229,0.1154,1.35


### 1.4 Prepare cleaned data for later sections
`load_split("train")`, `load_split("val")` and `load_split("test")` return all the cleaned trips of a split as one table, with the label in the `is_cash` column.

In [32]:
def load_split(name):
    paths = [PREPARED_DIR / f"{y}-{m:02d}.parquet" for y, m in SPLITS[name]]
    missing = [p.name for p in paths if not p.exists()]
    if missing:
        raise FileNotFoundError(f"{missing} not prepared yet: run Section 1 first")
    return pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)


train = load_split("train")
print(f"train: {len(train):,} trips, {train.shape[1]} columns, cash share {train[LABEL].mean():.3f}")
print("columns:", ", ".join(train.columns))

train: 49,406 trips, 18 columns, cash share 0.116
columns: VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, passenger_count, trip_distance, RatecodeID, store_and_fwd_flag, PULocationID, DOLocationID, fare_amount, extra, mta_tax, tolls_amount, improvement_surcharge, congestion_surcharge, airport_fee, cbd_congestion_fee, is_cash


## Section 2 · Features

Use the train and val split.

## Section 3 · Models

## Section 4 · Evaluation

Use the test split.